In [ ]:
import pandas as pd
from openai import OpenAI
import json

client = OpenAI()
 
# Read CSV into DataFrame
df = pd.read_csv("Corpus.csv",encoding="latin1") #Update with the name of the CSV with the sentences
 
# The sentence is in the second column
sentence_column = df.columns[0]  
sentences = df[sentence_column].dropna().tolist()
print(sentence_column)
results = []
 
for sentence in sentences:
    print("SENTENCE : " + sentence)
 
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": f"""You are analyzing sentences.  

                Your task is to analyze the given excerpt and classify words containing the stem “ancest” (case-insensitive). Follow these rules: 

                    Identification 

                        Extract every occurrence of any word that contains “ancest” (case-insensitive). 

                        Treat repeated instances of these words as separate occurrences. 

                        Ignore all words that do not contain “ancest” even if they are semantically related. 

                    Classification 

                        Assign exactly one category from the list below to each occurrence. 

                        Never assign multiple categories per occurrence. 

                        Do not infer or assume context beyond the explicit wording. Classify strictly based on the text itself. 

                Categories 

                    Genetic Ancestry and Health: how genetic ancestry influences health traits, disease risks, and biological pathways, linking ancestry-related genetic variation to medical outcomes. 

                    Ancestry in Population Structure and Genetic Variation: how ancestry shapes the organization, differentiation, and genetic relationships among populations, including stratification, clustering, and diversity patterns. 

                    Ancestry in Evolution and Lineage Relationships: how ancestry traces evolutionary history across species or lineages, including ancestral–descendant patterns and deep evolutionary connections.  

                    Human Ancestry, Identity, Geography, and Demography: how human ancestry relates to cultural identity, ethnic self-identification, migration history, regional distributions, and demographic patterns.  

                    Ancestry-Inference Tools, Markers, and Analytic Approaches: the genetic markers, mapping strategies, and analytical methods used to identify, quantify, or model ancestry.  

                    Unclear: use when the meaning cannot be confidently assigned to any of the above categories. 

                Output format: a JSON array, where each element is an object with: 

                {{ 

                    "word": "[[*ancest*]]", 

                    "label": "<category>", 

                    "justification": "<1–2 sentence reasoning>" 

                }} 

                Do NOT produce tables or markdown.  

                Sentence:
                {sentence}"""
            }
        ],
        temperature=0
    )

    label_json = completion.choices[0].message.content.strip()

    if label_json.startswith("```"):
        label_json = label_json.strip("`")
        label_json = label_json.replace("json\n", "", 1).strip()

    try:
        parsed_array = json.loads(label_json)
        results.append({
            "sentence": sentence,
            "labels": parsed_array  
        })
    except json.JSONDecodeError:
        results.append({
            "sentence": sentence,
            "labels": [{"word": "", "label": "Error", "justification": label_json}]
        })

with open("categorization.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

This protective haplotype has a frequency of 12% in the African ancestry, but only 0.003 in Europeans.
SENTENCE : This has been replicated in three independent African ancestry datasets and is trending in a much larger European dataset.
SENTENCE : Indeed, the ApoE e4 allele has a heterogeneous AD risk effect across diverse ancestral populations [3] (Fig 1).
SENTENCE : African (AF)-Ancestry populations (such as Ibadan individuals from Nigeria and African Americans (AA)) [3,610].
SENTENCE : Using admixed populations with the substantial proportion of AF ancestral genetic background (AA, Puerto Rico (PR) and the Dominican Republic), two independent studies [11,12] demonstrated that the difference in risk between AF and European (EU) populations lies in the ancestral genomic background surrounding the ApoE locus (local ancestry, or LA).
SENTENCE : Our objective is to follow up the local ancestry finding and identify the genetic variants that lower the risk for ApoE e4 in African ancestry.